
# 19-Introduction-to-SQL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/01-Foundations/19_Introduction_to_SQL.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=01-Foundations/19_Introduction_to_SQL.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [ ]:
# --- Global Notebook Setup ---
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Apply the standard course style for all plots
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)
%config InlineBackend.figure_format = 'retina'  # High-res plots

np.set_printoptions(suppress=True, linewidth=120, precision=4)
pd.set_option("display.float_format", "{:.2f}".format)
warnings.filterwarnings("ignore", category=FutureWarning)


### Table of Contents
1. [The Lens: SQL as the Lingua Franca of Data](#The-Lens:-SQL-as-the-Lingua-Franca-of-Data)
2. [The Relational Model: Tables, Keys, and Relationships](#The-Relational-Model:-Tables,-Keys,-and-Relationships)
3. [Setting Up: SQLite and Python](#Setting-Up:-SQLite-and-Python)
4. [Basic Querying: `SELECT` and `WHERE`](#Basic-Querying:-SELECT-and-WHERE)
5. [Aggregating Data: `GROUP BY` and `HAVING`](#Aggregating-Data:-GROUP-BY-and-HAVING)
6. [Combining Tables: The Art of the `JOIN`](#Combining-Tables:-The-Art-of-the-JOIN)
7. [Advanced SQL: Subqueries and Window Functions](#Advanced-SQL:-Subqueries-and-Window-Functions)
8. [Summary](#Summary)
9. [Exercises](#Exercises)


## The Lens: 19-Introduction-to-SQL
While Pandas is powerful for analysis, data rarely originates in a CSV file. In the real world, especially in policy institutions, banks, and tech firms, data lives in **Relational Database Management Systems (RDBMS)**. 

**Structured Query Language (SQL)** is the universal language for interacting with these systems. It is not just a way to retrieve data; it is a declarative programming paradigm for data manipulation. Unlike Python (imperative), where you tell the computer *how* to do something, in SQL you tell the database *what* you want, and its query optimizer figures out the most efficient way to get it.

For an economist, SQL is essential for:
1.  **Extracting** subsets of massive datasets that are too large to fit in memory.
2.  **Joining** disparate datasets (e.g., merging census data with tax records).
3.  **Aggregating** transactional data into a usable form for analysis.

This notebook introduces SQL using **SQLite**, a lightweight, file-based database engine included with Python, making it perfect for learning without complex server setup.
### Learning Objectives
* **Write** SQL queries: SELECT, WHERE, JOIN, GROUP BY, and subqueries.
* **Interact** with SQLite databases from Python using `sqlite3` and Pandas.
* **Design** normalized database schemas and understand relational data modeling.

### Prerequisites
* **Pandas:** DataFrame operations, merges, and groupby (Module 01 - Pandas).
* **Python Basics:** String formatting and basic file I/O.
* **Learning-path prerequisite:** [`18_Data_Acquisition_Web_Scraping.ipynb`](18_Data_Acquisition_Web_Scraping.ipynb)


> **Learning path:** Building on [`18_Data_Acquisition_Web_Scraping.ipynb`](18_Data_Acquisition_Web_Scraping.ipynb); next continue with [`20_Introduction_to_SciPy.ipynb`](20_Introduction_to_SciPy.ipynb).


### The Relational Model: Tables, Keys, and Relationships

A relational database organizes data into **tables** (relations). 
- **Rows** represent observations (records).
- **Columns** represent attributes (variables).
- **Primary Key**: A unique identifier for each row in a table (e.g., `student_id`).
- **Foreign Key**: A field that links to the primary key of another table, establishing a relationship.


### Setting Up: SQLite and Python

We will use the `sqlite3` library to create an in-memory database and populate it with sample economic data.


In [ ]:
# Create a connection to an in-memory database
conn = sqlite3.connect(":memory:")

# Create sample DataFrames
countries_data = {
    "country_code": ["USA", "CHN", "JPN", "DEU", "GBR"],
    "name": ["United States", "China", "Japan", "Germany", "United Kingdom"],
    "continent": ["North America", "Asia", "Asia", "Europe", "Europe"],
}

gdp_data = {
    "country_code": ["USA", "USA", "CHN", "CHN", "JPN", "JPN", "DEU", "DEU"],
    "year": [2020, 2021, 2020, 2021, 2020, 2021, 2020, 2021],
    "gdp_trillions": [20.89, 23.00, 14.72, 17.73, 5.06, 4.94, 3.85, 4.22],
}

df_countries = pd.DataFrame(countries_data)
df_gdp = pd.DataFrame(gdp_data)

# Write DataFrames to SQL tables
df_countries.to_sql("countries", conn, index=False)
df_gdp.to_sql("gdp", conn, index=False)

print("Database initialized with tables: 'countries' and 'gdp'.")


### Basic Querying: `SELECT` and `WHERE`

The most fundamental SQL operation is the `SELECT` statement. Its structure is:
```sql
SELECT columns 
FROM table 
WHERE condition;
```


In [ ]:
# Query: Select all columns for GDP records where GDP > 15 trillion
query = """
SELECT *
FROM gdp
WHERE gdp_trillions > 15.0
"""

result = pd.read_sql(query, conn)
display(result)


### Aggregating Data: `GROUP BY` and `HAVING`

Aggregation allows us to compute summary statistics (sum, average, count) for groups of rows.

- `GROUP BY`: Group rows that have the same values in specified columns.
- `HAVING`: Filter groups based on the result of an aggregate function (like `WHERE`, but for groups).


In [ ]:
# Query: Calculate average GDP for each country
query = """
SELECT country_code, AVG(gdp_trillions) as avg_gdp
FROM gdp
GROUP BY country_code
ORDER BY avg_gdp DESC
"""

result = pd.read_sql(query, conn)
display(result)


### Combining Tables: The Art of the `JOIN`

The real power of relational databases lies in `JOIN`s. They allow you to combine data from multiple tables based on a related column (key).

- **`INNER JOIN`**: Returns records that have matching values in both tables.
- **`LEFT JOIN`**: Returns all records from the left table, and the matched records from the right table. (Useful for preserving observations).


In [ ]:
# Query: Join GDP data with Country metadata to get continent info
query = """
SELECT 
    g.year, 
    c.name as country_name, 
    c.continent, 
    g.gdp_trillions
FROM gdp g
INNER JOIN countries c ON g.country_code = c.country_code
WHERE g.year = 2021
ORDER BY g.gdp_trillions DESC
"""

result = pd.read_sql(query, conn)
display(result)


### Advanced SQL: Subqueries and Window Functions

Modern SQL includes powerful analytical tools.

- **Subqueries**: Queries nested inside other queries.
- **Window Functions**: Perform calculations across a set of table rows that are somehow related to the current row (e.g., rolling averages, rankings) without collapsing the result like `GROUP BY`.

Example: Calculating the year-over-year GDP growth rate using `LAG()`.


In [ ]:
query = """
SELECT 
    country_code,
    year,
    gdp_trillions,
    LAG(gdp_trillions) OVER (PARTITION BY country_code ORDER BY year) as prev_year_gdp,
    (gdp_trillions - LAG(gdp_trillions) OVER (PARTITION BY country_code ORDER BY year)) / 
    LAG(gdp_trillions) OVER (PARTITION BY country_code ORDER BY year) * 100 as growth_rate
FROM gdp
"""

result = pd.read_sql(query, conn)
display(result)


#### Common Table Expressions (CTEs)
CTEs (defined using `WITH`) allow you to create temporary result sets that can be referenced within a larger query. This improves readability and modularity, acting like functions in SQL.

```sql
WITH RecentGDP AS (
    SELECT country_code, gdp_trillions
    FROM gdp
    WHERE year = 2021
)
SELECT *
FROM RecentGDP
WHERE gdp_trillions > 10;
```
While SQLite supports CTEs, their power is most evident in complex queries involving multiple stages of transformation.


### Writing Safe Queries: Injection and Parameterized Statements

A classic security disaster is building a query with a raw string interpolation of user input. Imagine a login query assembled as

```python
query = f"SELECT * FROM users WHERE name = '{user_input}'"
```

If `user_input` is `x' OR '1'='1`, the executed statement becomes `SELECT * FROM users WHERE name = 'x' OR '1'='1'` — the WHERE clause is now always true and every row is returned. This is a **SQL injection attack**, and it is entirely preventable: never interpolate values into SQL text. Pass them as *parameters* and let the driver bind them safely.


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE users (name TEXT, country TEXT)")
cur.executemany("INSERT INTO users VALUES (?, ?)",
                [("alice", "US"), ("bob", "DE"), ("carol", None)])

# --- The WRONG way: interpolating input into the query string ---
user_input = "x' OR '1'='1"
vulnerable = "SELECT * FROM users WHERE name = '%s'" % user_input
print("Vulnerable query:", vulnerable)
print("Rows returned:", len(cur.execute(vulnerable).fetchall()),
      "(every row!)")

# --- The RIGHT way: parameterized query ---
safe = "SELECT * FROM users WHERE name = ?"
row = cur.execute(safe, (user_input,)).fetchall()
print("Parameterized query:", safe)
print("Rows returned:", len(row), "(the injected text is treated as data)")


### `NULL`: Three-Valued Logic in Practice

`NULL` means *unknown*, not zero or an empty string. Two consequences bite every analyst eventually:

1. **Comparisons with `NULL` yield `NULL`, not true or false** — so `WHERE x = NULL` matches nothing. Use `IS NULL` / `IS NOT NULL`.
2. **Aggregates skip `NULL`s**: `COUNT(*)` counts rows, `COUNT(col)` counts only non-null values — dividing one by the other is how you measure missingness.


In [ ]:
# NULL semantics demo on the users table
print("COUNT(*)   =", cur.execute("SELECT COUNT(*) FROM users").fetchone()[0])
print("COUNT(country) =",
      cur.execute("SELECT COUNT(country) FROM users").fetchone()[0],
      "-> carol's missing country is not counted")

# '= NULL' matches nothing; 'IS NULL' does
print("rows with country = NULL:",
      cur.execute("SELECT COUNT(*) FROM users WHERE country = NULL").fetchone()[0])
print("rows with country IS NULL:",
      cur.execute("SELECT COUNT(*) FROM users WHERE country IS NULL").fetchone()[0])

conn.close()


### Three-Tier Practice Ladder

**1. Mechanism and assumptions (Conceptual):** Explain the central computational idea in **19-Introduction-to-SQL** and connect it to one explicit economic object or research workflow.

**2. Reproduce and diagnose (Applied):** Reproduce an example involving The Relational Model: Tables, Keys, and Relationships, Setting Up: SQLite and Python, then change one input and explain the result before running the code.

**3. Robust extension (Challenge):** Extend the example to a larger or less convenient case and document the correctness and performance checks needed before trusting the result.

> Use the existing exercises above when they target the same skill; this ladder makes the intended progression explicit rather than replacing instructor-authored problems.


# Summary

SQL is the foundation of data engineering. We've covered:
- **The Relational Model**: Understanding tables and keys.
- **Querying**: `SELECT`, `WHERE` for filtering.
- **Aggregation**: `GROUP BY` for summary statistics.
- **Joins**: Combining data from multiple sources.
- **Window Functions**: Advanced analytics like growth rates directly in the database.

By offloading data processing to the database layer, you can handle datasets far larger than your machine's memory, pulling only the refined results into Python for final analysis.


### Exercises

1.  **Continent Aggregation:** Write a query to calculate the total GDP for each continent in the year 2021.
2.  **Filtering with Subquery:** Find all countries whose 2021 GDP was above the global average GDP for that year.
3.  **Rankings:** Use the `RANK()` window function to rank countries by GDP within each continent for the year 2020.


## References & Further Reading

- Python Software Foundation. *Python 3 Documentation*.
- Harris, C. R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362.
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly.
